In [1]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


In [2]:

# =========================
# 1) Load RFECV-selected dataset
# =========================
df = pd.read_csv("../wrapper/fraud_selected_features_rfecv.csv")

target = "fraud"
X = df.drop(columns=[target])
y = df[target].astype(int)


In [3]:

# =========================
# 2) Standardize features
#    PCA is sensitive to scale
# =========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [4]:

# =========================
# 3) Apply PCA
#    Keep enough components to explain 95% variance
# =========================
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)


In [5]:

# =========================
# 4) Build PCA dataframe
# =========================
pc_columns = [f"PC{i+1}" for i in range(X_pca.shape[1])]
pca_df = pd.DataFrame(X_pca, columns=pc_columns, index=df.index)
pca_df[target] = y.values

OUT_DIR = Path("../PCA")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save transformed dataset
pca_df.to_csv("../PCA/fraud_pca_95_variance.csv", index=False)


In [6]:

# =========================
# 5) Print summary
# =========================
print("Original selected feature count :", X.shape[1])
print("Number of PCA components kept  :", X_pca.shape[1])
print("Explained variance ratio per component:")
for i, var in enumerate(pca.explained_variance_ratio_, start=1):
    print(f"PC{i}: {var:.4f}")

print("\nCumulative explained variance:")
cum_var = pca.explained_variance_ratio_.cumsum()
for i, var in enumerate(cum_var, start=1):
    print(f"PC1 to PC{i}: {var:.4f}")

print("\nPCA dataset saved as: fraud_pca_95_variance.csv")

Original selected feature count : 17
Number of PCA components kept  : 11
Explained variance ratio per component:
PC1: 0.2025
PC2: 0.1695
PC3: 0.1055
PC4: 0.0990
PC5: 0.0622
PC6: 0.0603
PC7: 0.0592
PC8: 0.0591
PC9: 0.0587
PC10: 0.0583
PC11: 0.0274

Cumulative explained variance:
PC1 to PC1: 0.2025
PC1 to PC2: 0.3721
PC1 to PC3: 0.4776
PC1 to PC4: 0.5765
PC1 to PC5: 0.6387
PC1 to PC6: 0.6990
PC1 to PC7: 0.7582
PC1 to PC8: 0.8172
PC1 to PC9: 0.8759
PC1 to PC10: 0.9342
PC1 to PC11: 0.9616

PCA dataset saved as: fraud_pca_95_variance.csv


In [7]:

loadings = pd.DataFrame(
    pca.components_.T,
    index=X.columns,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)]
)

print(loadings)
loadings.to_csv("../PCA/fraud_pca_loadings.csv")
print("PCA loadings saved as: fraud_pca_loadings.csv")

                                         PC1       PC2       PC3       PC4  \
Latitude                           -0.010687 -0.564723  0.006007  0.000738   
order_weekday                      -0.001801  0.009079  0.700178 -0.094021   
Category Name__freq                 0.529872 -0.009626  0.002723  0.000351   
Product Image__freq                 0.532277 -0.009756  0.003161  0.000201   
Product Name__freq                  0.532277 -0.009756  0.003161  0.000201   
order_is_weekend                   -0.003644  0.008627  0.700807 -0.088681   
Customer Zipcode_was_missing       -0.006232 -0.002768  0.002783  0.000605   
Customer Country_EE. UU.           -0.010648 -0.583327  0.007408 -0.003801   
Customer Country_Puerto Rico        0.010648  0.583327 -0.007408  0.003801   
Customer Segment_Consumer          -0.000896 -0.005363  0.089887  0.701302   
Customer Segment_Corporate          0.001210 -0.001953 -0.092587 -0.700958   
Department Name_Book Shop          -0.064221  0.001526 -0.010802